In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import warnings

warnings.filterwarnings('ignore')

In [3]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
original = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [4]:
train['Churn'] = train['Churn'].map({'Yes': 1, 'No': 0}).astype(int)
original['Churn'] = original['Churn'].map({'Yes': 1, 'No': 0}).astype(int)

original['TotalCharges'] = pd.to_numeric(original['TotalCharges'], errors='coerce')
original['TotalCharges'].fillna(original['TotalCharges'].median(), inplace=True)
# original['TotalCharges'].fillna(original['MonthlyCharges'] * original['tenure'], inplace=True)

In [5]:
original.drop('customerID', axis=1, inplace=True)
train.drop('id', axis=1, inplace=True)
test.drop('id', axis=1, inplace=True)

In [6]:
cats = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

nums = ['tenure', 'MonthlyCharges', 'TotalCharges']

target = 'Churn'

new_nums = []
nums_as_cat = []

In [7]:
for col in nums:
    freq = pd.concat([train[col], original[col], test[col]]).value_counts(normalize=True)
    for df in [train, test, original]:
        df[f'Freq{col}'] = df[col].map(freq).fillna(0).astype('float32')
    new_nums.append(f'Freq{col}')

In [8]:
for col in cats:

    churn_mean = original.groupby(col)[target].mean()
    churn_std = original.groupby(col)[target].std()

    train[f"{col}ChurnMean"] = train[col].map(churn_mean)
    test[f"{col}ChurnMean"] = test[col].map(churn_mean)

    train[f"{col}ChurnStd"] = train[col].map(churn_std)
    test[f"{col}ChurnStd"] = test[col].map(churn_std)

In [9]:
q1 = train["MonthlyCharges"].quantile(0.25)
q3 = train["MonthlyCharges"].quantile(0.75)

for df in [train, test]:

    df["ChargeOutlier"] = (
        (df["MonthlyCharges"] < q1) |
        (df["MonthlyCharges"] > q3)
    ).astype(int)

    df['MCLog'] = np.log1p(df['MonthlyCharges'])
    df['TCLog'] = np.log1p(df['TotalCharges'])

    df['MCSqueezed'] = 1 / (df['MonthlyCharges'] + 1)
    df['TCSqueezed'] = 1 / (df['TotalCharges'] + 1)

    df["TenureLog"] = np.log1p(df["tenure"])
    df['TenureSqueezed'] = 1 / (df['tenure'] + 1)

    df["AvgMonthlySpend"] = df["TotalCharges"] / (df["tenure"] + 1)
    df["SpendDeviation"] = df["MonthlyCharges"] - df["AvgMonthlySpend"]
    df["MonthlyToTotalRatio"] = df["MonthlyCharges"] / (df["TotalCharges"] + 1)

    df["IsMonthToMonth"] = (df["Contract"] == "Month-to-month").astype(int)

    df["IsElectronicCheck"] = (df["PaymentMethod"] == "Electronic check").astype(int)
    df["AutoPayment"] = df["PaymentMethod"].str.contains("automatic").astype(int)

    df["ServiceCount"] = (
        (df["OnlineSecurity"] == "Yes").astype(int) +
        (df["OnlineBackup"] == "Yes").astype(int) +
        (df["DeviceProtection"] == "Yes").astype(int) +
        (df["TechSupport"] == "Yes").astype(int) +
        (df["StreamingTV"] == "Yes").astype(int) +
        (df["StreamingMovies"] == "Yes").astype(int)
    )

    df["IsFiber"] = (df["InternetService"] == "Fiber optic").astype(int)

    df["FiberHighCharge"] = (
        (df["InternetService"] == "Fiber optic") &
        (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    ).astype(int)

    df["FiberNoSupport"] = (
        (df["InternetService"] == "Fiber optic") &
        (df["TechSupport"] == "No")
    ).astype(int)

    df["EarlyTenure"] = (df["tenure"] < 12).astype(int)
    df["HighChargeEarly"] = (
        (df["tenure"] < 12) &
        (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    ).astype(int)

    df["TenureBucket"] = pd.cut(
        df["tenure"],
        bins=[0, 6, 12, 24, 48, 72],
        labels=False
    )

    df["VeryEarly"] = (df["tenure"] < 6).astype(int)

    df["LifetimeValue"] = df["MonthlyCharges"] * df["tenure"]

    df["ChargeRank"] = df["MonthlyCharges"].rank(pct=True)
    df["TenureRank"] = df["tenure"].rank(pct=True)

new_nums = [col for col in df.columns.tolist() if col not in (nums + cats + [target])]

In [10]:
for col in cats + nums:
    tmp = original.groupby(col)[target].mean()
    name = f"OrigP{col}"
    train = train.merge(tmp.rename(name), on=col, how="left")
    test = test.merge(tmp.rename(name), on=col, how="left")
    for df in [train, test]:
        df[name] = df[name].fillna(0.5).astype('float32')
    new_nums.append(name)

In [11]:
X, y = train.drop(target, axis=1), train[target]

In [12]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

label_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling'
]

onehot_cols = ['PaymentMethod']

ord_mapping = {
    'Contract': ['Two year', 'One year', 'Month-to-month'],
    'MultipleLines': ['No phone service', 'No', 'Yes'],
    'InternetService': ['No', 'DSL', 'Fiber optic'],
    'OnlineSecurity': ['No internet service', 'No', 'Yes'],
    'OnlineBackup': ['No internet service', 'No', 'Yes'],
    'DeviceProtection': ['No internet service', 'No', 'Yes'],
    'TechSupport': ['No internet service', 'No', 'Yes'],
    'StreamingTV': ['No internet service', 'No', 'Yes'],
    'StreamingMovies': ['No internet service', 'No', 'Yes'],
    'Partner': ['No', 'Yes'],
    'Dependents': ['No', 'Yes'],
    'PhoneService': ['No', 'Yes'],
    'PaperlessBilling': ['No', 'Yes'],
    'gender': ['Male', 'Female']
}

ord_cols = list(ord_mapping.keys())

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', sparse_output=False), onehot_cols),
        ('ordinal', OrdinalEncoder(categories=[ord_mapping[col] for col in ord_cols]), ord_cols)
    ],
    remainder='passthrough'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor)
])

pipeline.set_output(transform="pandas")

X_train_encoded = pipeline.fit_transform(X)
X_test_encoded = pipeline.transform(test)

In [39]:
import json

with open('xgboost-params.json', 'r') as file:
    xgb_params = json.load(file)

In [40]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=20,
    shuffle=True,
    random_state=420
)

oof_preds = np.zeros(len(X_train_encoded))
test_preds = np.zeros(len(X_test_encoded))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_encoded, y)):

    print(f'Fold {fold+1}')

    X_train, X_valid = X_train_encoded.iloc[train_idx], X_train_encoded.iloc[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]

    model = XGBClassifier(
        n_estimators=20000,
        eval_metric='auc',
        random_state=420,
        early_stopping_rounds=500,
        **xgb_params
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        verbose=200
    )

    oof_preds[valid_idx] = model.predict_proba(X_valid)[:, 1]
    test_preds += model.predict_proba(X_test_encoded)[:, 1] / skf.n_splits

Fold 1
[0]	validation_0-auc:0.89325
[200]	validation_0-auc:0.91135
[400]	validation_0-auc:0.91335
[600]	validation_0-auc:0.91410
[800]	validation_0-auc:0.91458
[1000]	validation_0-auc:0.91496
[1200]	validation_0-auc:0.91528
[1400]	validation_0-auc:0.91551
[1600]	validation_0-auc:0.91569
[1800]	validation_0-auc:0.91583
[2000]	validation_0-auc:0.91598
[2200]	validation_0-auc:0.91607
[2400]	validation_0-auc:0.91617
[2600]	validation_0-auc:0.91626
[2800]	validation_0-auc:0.91631
[3000]	validation_0-auc:0.91635
[3200]	validation_0-auc:0.91640
[3400]	validation_0-auc:0.91643
[3600]	validation_0-auc:0.91647
[3800]	validation_0-auc:0.91652
[4000]	validation_0-auc:0.91655
[4200]	validation_0-auc:0.91659
[4400]	validation_0-auc:0.91661
[4600]	validation_0-auc:0.91666
[4800]	validation_0-auc:0.91667
[5000]	validation_0-auc:0.91669
[5200]	validation_0-auc:0.91670
[5400]	validation_0-auc:0.91672
[5600]	validation_0-auc:0.91674
[5800]	validation_0-auc:0.91674
[6000]	validation_0-auc:0.91675
[6200]	v

In [41]:
from sklearn.metrics import roc_auc_score

cv_score = roc_auc_score(y, oof_preds)
print("CV ROC-AUC:", cv_score)

CV ROC-AUC: 0.9176262837754605


In [42]:
submission = pd.read_csv('sample_submission.csv')
submission['Churn'] = test_preds
submission.to_csv('submission.csv', index=False)

In [43]:
with open('lightgbm-params.json', 'r') as file:
    params = json.load(file)

In [45]:
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=20,
    shuffle=True,
    random_state=420
)

oof_preds = np.zeros(len(X_train_encoded))
test_preds = np.zeros(len(X_test_encoded))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_encoded, y)):

    print(f'Fold {fold+1}')

    X_train, X_valid = X_train_encoded.iloc[train_idx], X_train_encoded.iloc[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]

    model = LGBMClassifier(
        objective="binary",
        metric="auc",
        n_estimators=20000,
        verbosity=-1,
        random_state=420,
        **params
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
        callbacks=[lgb.log_evaluation(100),
        lgb.early_stopping(500)] 
    )

    oof_preds[valid_idx] = model.predict_proba(X_valid)[:, 1]
    test_preds += model.predict_proba(X_test_encoded)[:, 1] / skf.n_splits

Fold 1
Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.913447
[200]	valid_0's auc: 0.914674
[300]	valid_0's auc: 0.915289
[400]	valid_0's auc: 0.915556
[500]	valid_0's auc: 0.9158
[600]	valid_0's auc: 0.916055
[700]	valid_0's auc: 0.916181
[800]	valid_0's auc: 0.916244
[900]	valid_0's auc: 0.916373
[1000]	valid_0's auc: 0.916448
[1100]	valid_0's auc: 0.916485
[1200]	valid_0's auc: 0.916486
[1300]	valid_0's auc: 0.916572
[1400]	valid_0's auc: 0.916592
[1500]	valid_0's auc: 0.916603
[1600]	valid_0's auc: 0.916605
[1700]	valid_0's auc: 0.916589
Early stopping, best iteration is:
[1556]	valid_0's auc: 0.916625
Fold 2
Training until validation scores don't improve for 200 rounds
[100]	valid_0's auc: 0.917393
[200]	valid_0's auc: 0.918506
[300]	valid_0's auc: 0.919038
[400]	valid_0's auc: 0.919265
[500]	valid_0's auc: 0.919394
[600]	valid_0's auc: 0.919486
[700]	valid_0's auc: 0.919613
[800]	valid_0's auc: 0.919715
[900]	valid_0's auc: 0.919804
[1000]	val

In [46]:
cv_score = roc_auc_score(y, oof_preds)
print("CV ROC-AUC:", cv_score)

CV ROC-AUC: 0.9175289375865667


In [47]:
submission = pd.read_csv('sample_submission.csv')
submission['Churn'] = test_preds
submission.to_csv('submission.csv', index=False)